# Rainfall objects in the Magdalena River Basin (daily): volume aggregation

Rainfall events are 3-D (time, latitude, longitude) connected objects of cells at or above a rainfall threshold. For events that start or end during La Nina or El Nino months, this notebook computes volume, peak rainfall, centroids and elevations. The calculations live in `snlda.objects`.

## 0. Setup

In [ ]:
from pathlib import Path

DATA = Path("Data")
RESULTS = Path("Results")

STATIONS_CSV = DATA / "Local CSV" / "Precipitation 1980-2020 data filled 1.csv"
COORDS_CSV = DATA / "Local CSV" / "Precipitation 1980-2020 coord filled.csv"
IDW_NC = DATA / "Local IDW" / "IDW_daily_010.nc"
ERA_NC = DATA / "ERALand NC" / "precipitation_eraland_d2.nc"
BASIN_NC = DATA / "Magdalena" / "GIS" / "Basin" / "Basin_ext_010.nc"

ONI_CSV = DATA / "Magdalena" / "ENSO" / "ONI.csv"
GEOPOTENTIAL_NC = DATA / "Magdalena" / "Reanalysis" / "ERA5" / "Geopotential" / "geopotential.nc"

THRESHOLD = 1.0      # mm: what counts as rain
MAX_EVENTS = 1000    # keep the largest events per ENSO phase

import numpy as np

from snlda import grid, io, objects

## 1. Load data

In [ ]:
idw = io.read_netcdf_grid(IDW_NC)
era = io.read_netcdf_grid(ERA_NC)
era.data[era.data < 0] = np.nan        # no-data values
basin = io.read_basin_mask(BASIN_NC)
oni = io.read_oni(ONI_CSV)
z, z_lat, z_lon = io.read_geopotential(GEOPOTENTIAL_NC)

nina = objects.enso_months(oni, "nina")
nino = objects.enso_months(oni, "nino")

## 2. Events per dataset and ENSO phase

In [ ]:
out_dir = RESULTS / "Volume Aggregation"
out_dir.mkdir(parents=True, exist_ok=True)

datasets = {"IDW10": idw, "eraland": era}

for name, ds in datasets.items():
    cropped = grid.crop_to_basin(ds.data, ds.lat, ds.lon, basin.mask, basin.lat, basin.lon)
    elevation = grid.elevation_on_grid(z, z_lat, z_lon, cropped.lat, cropped.lon)
    rain, labels, n_objects = objects.rain_objects(cropped.data, threshold=THRESHOLD)
    print(f"{name}: {n_objects} rainfall objects")

    for phase, months in (("nina", nina), ("nino", nino)):
        events = objects.enso_event_table(
            labels, rain, ds.time, cropped.lat, cropped.lon, months,
            elevation=elevation, max_events=MAX_EVENTS,
        )
        print(f"  {phase}: {len(events)} events")
        print(events["area"].quantile([0.5, 0.9, 0.99, 1.0]).to_string())
        events.to_csv(out_dir / f"{name}_daily_{phase}2.csv")